# Advanced Retrieval Methods: Comparative Evaluation

## Objective

This notebook provides a comprehensive evaluation of various retrieval strategies using the LangChain framework. We compare six different retriever implementations across two chunking strategies:

**Retriever Methods:**
1. Naive Retrieval (Baseline)
2. BM25 Retrieval
3. Contextual Compression (Reranking)
4. Multi-Query Retrieval
5. Parent Document Retrieval
6. Ensemble Retrieval

**Chunking Strategies:**
1. Standard Chunking (Fixed-size with RecursiveCharacterTextSplitter)
2. Semantic Chunking (Content-aware boundary detection)

## Evaluation Methodology

We use Ragas (Retrieval Augmented Generation Assessment) framework to:
- Generate synthetic test datasets with ground truth
- Measure retriever performance using standard metrics:
  - Context Precision
  - Context Recall
  - Context Relevancy
- Track cost and latency via LangSmith

## Dataset

The evaluation uses a corpus of 50 AI/ML project descriptions from the AIE bootcamp, including project metadata such as domains, scores, and judge feedback.


---

## Section 1: Environment Setup and Dependencies


In [1]:
# Core Python libraries
import os
import pandas as pd
from datetime import datetime
from typing import List, Dict, Any
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Verify required API keys
required_keys = ["OPENAI_API_KEY", "COHERE_API_KEY", "LANGCHAIN_API_KEY"]
missing_keys = [key for key in required_keys if not os.getenv(key)]

if missing_keys:
    raise ValueError(f"Missing API keys: {', '.join(missing_keys)}")

print(f"API keys loaded | LangSmith tracing: {os.getenv('LANGCHAIN_TRACING_V2', 'false')}")


API keys loaded | LangSmith tracing: true


### Import Dependencies


In [2]:
# LangChain Core
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter

# LangChain Document Loaders and Text Splitters
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker

# LangChain Models
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# LangChain Vector Stores
from langchain_community.vectorstores import Qdrant
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient, models

# LangChain Retrievers
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.retrievers import ParentDocumentRetriever, EnsembleRetriever
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain.storage import InMemoryStore

# Cohere for Reranking
from langchain_cohere import CohereRerank

# Ragas for Evaluation (v0.3.x)
from ragas.testset import TestsetGenerator
from ragas.testset.synthesizers import (
    SingleHopSpecificQuerySynthesizer,
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer
)
from ragas import evaluate
from ragas.metrics import context_precision, context_recall, ContextRelevance

print("All dependencies imported successfully")


All dependencies imported successfully


### Initialize Models and Embeddings


In [3]:
# Initialize models
chat_model = ChatOpenAI(model="gpt-4.1-nano", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print(f"Models initialized: {chat_model.model_name} | {embeddings.model}")


Models initialized: gpt-4.1-nano | text-embedding-3-small


---

## Implementation Plan

### Overview

This evaluation compares **12 retriever configurations**:
- 6 retriever methods × 2 chunking strategies = 12 total evaluations

### Execution Phases

#### Phase 1: Setup and Data Preparation
1. **Environment Configuration**
   - Load dependencies (LangChain, Ragas, Cohere, Qdrant)
   - Load API keys from .env file (OpenAI, Cohere, LangSmith)
   - Enable LangSmith tracing for cost/latency tracking

2. **Data Loading**
   - Load "howpeopleuseai.pdf" from data folder
   - Convert to LangChain Document format using PyPDFLoader
   - Verify document structure and content

3. **Document Chunking**
   - Create standard chunks using RecursiveCharacterTextSplitter
   - Create semantic chunks using SemanticChunker
   - Compare chunk statistics

4. **Vector Store Creation**
   - Build Qdrant vector stores for both chunking strategies
   - Configure embeddings (OpenAI text-embedding-3-small)
   - Verify vector store operations

#### Phase 2: Golden Dataset Generation
5. **Synthetic Test Generation**
   - Use Ragas TestsetGenerator with document-based approach
   - Generate 15 test questions with ground truth
   - Include diverse question types (simple, reasoning, multi-context)
   - Validate generated questions

#### Phase 3: Retriever Implementation
6. **Standard Chunking Retrievers**
   - Implement all 6 retrievers using standard chunks
   - Configure parameters to match baseline notebook

7. **Semantic Chunking Retrievers**
   - Implement all 6 retrievers using semantic chunks
   - Maintain consistent configuration

#### Phase 4: Evaluation Execution
8. **Run Evaluations**
   - Execute each of 12 configurations against golden dataset (15 questions)
   - Collect retriever-appropriate Ragas metrics per assignment requirements
   - Track execution time per retriever

9. **Cost and Latency Analysis**
   - Extract metrics from LangSmith traces
   - Calculate average cost per query
   - Measure average latency per retriever

#### Phase 5: Analysis and Reporting
10. **Comparative Analysis**
    - Create results comparison table
    - Analyze chunking strategy impact
    - Identify best-performing retrievers
    
11. **Final Recommendations**
    - Synthesize findings
    - Recommend optimal configuration
    - Discuss tradeoffs (performance vs cost vs latency)

### Expected Outputs

1. Comprehensive metrics table
2. Chunking strategy comparison
3. Cost-benefit analysis
4. Performance recommendations
5. Implementation insights


---

## Section 2: Data Loading


In [4]:
# Load all PDF documents from data folder
import glob

pdf_files = glob.glob("./data/*.pdf")
documents = []

for pdf_path in pdf_files:
    loader = PyMuPDFLoader(pdf_path)
    docs = loader.load()
    documents.extend(docs)

print(f"Loaded {len(pdf_files)} PDFs: {len(documents)} total pages")


Loaded 1 PDFs: 64 total pages


---

## Section 3: Golden Dataset Generation

Generate synthetic test questions using Ragas TestsetGenerator. This step creates the evaluation dataset before building retrieval infrastructure.


In [5]:
#creating model and embedding for testset generation
generator_model = ChatOpenAI(model="gpt-4.1-nano")
generator_embedding = OpenAIEmbeddings()

# Generate synthetic test dataset using Ragas v0.3.x
# Wrap LLM and embeddings for Ragas compatibility
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

generator_llm = LangchainLLMWrapper(chat_model)
generator_embedding = LangchainEmbeddingsWrapper(embeddings)

generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embedding
)

# Generate 15 test questions
testset = generator.generate_with_langchain_docs(
    documents,
    testset_size=15
)

print(f"Generated {len(testset.samples)} test questions")


/var/folders/cz/01ttpj991sx782zbyzgb4bdc0000gn/T/ipykernel_7308/3024679437.py:10: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use the modern LLM providers instead: from ragas.llms.base import llm_factory; llm = llm_factory('gpt-4o-mini') or from ragas.llms.base import instructor_llm_factory; llm = instructor_llm_factory('openai', client=openai_client)
  generator_llm = LangchainLLMWrapper(chat_model)
/var/folders/cz/01ttpj991sx782zbyzgb4bdc0000gn/T/ipykernel_7308/3024679437.py:11: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embedding = LangchainEmbeddingsWrapper(embeddings)


Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/31 [00:00<?, ?it/s]

Property 'summary' already exists in node '597214'. Skipping!
Property 'summary' already exists in node '0793ec'. Skipping!
Property 'summary' already exists in node '84874e'. Skipping!
Property 'summary' already exists in node '22908f'. Skipping!
Property 'summary' already exists in node '7a795b'. Skipping!
Property 'summary' already exists in node 'd23d0f'. Skipping!
Property 'summary' already exists in node '8183ad'. Skipping!
Property 'summary' already exists in node '77e1de'. Skipping!
Property 'summary' already exists in node 'f487b8'. Skipping!
Property 'summary' already exists in node 'f80d21'. Skipping!


...

Applying CosineSimilarityBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Applying OverlapScoreBuilder:   0%|          | 0/1 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

ValidationError: 1 validation error for ThemesPersonasInput
themes.0
  Input should be a valid string [type=string_type, input_value=('Ouyang', 'Ouyang'), input_type=tuple]
    For further information visit https://errors.pydantic.dev/2.11/v/string_type

In [ ]:
# Convert to DataFrame for easy inspection
golden_dataset = testset.to_pandas()

# Display sample
print(f"\nSample questions:")
for i in range(min(3, len(golden_dataset))):
    print(f"\nQ{i+1}: {golden_dataset.iloc[i]['user_input']}")
    if 'reference' in golden_dataset.columns:
        print(f"Reference: {str(golden_dataset.iloc[i]['reference'])[:100]}...")
    
print(f"\nDataset shape: {golden_dataset.shape}")
print(f"Columns: {list(golden_dataset.columns)}")


---

## Section 3: Document Chunking

We create two sets of chunks from the same source documents to compare retrieval performance:

1. **Standard Chunking**: Fixed-size chunks using RecursiveCharacterTextSplitter
2. **Semantic Chunking**: Content-aware chunks using SemanticChunker


### Strategy 1: Standard Chunking


In [ ]:
# Create token-based text splitter
import tiktoken

encoder = tiktoken.encoding_for_model("gpt-4.1-nano")

standard_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # tokens
    chunk_overlap=50,  # tokens
    length_function=lambda text: len(encoder.encode(text))
)

standard_chunks = standard_splitter.split_documents(documents)

print(f"Standard chunks: {len(standard_chunks)} (avg: {sum(len(encoder.encode(c.page_content)) for c in standard_chunks) / len(standard_chunks):.0f} tokens)")


### Strategy 2: Semantic Chunking


In [ ]:
# Create semantic chunker
semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

semantic_chunks = semantic_chunker.split_documents(documents)

print(f"Semantic chunks: {len(semantic_chunks)} (avg: {sum(len(c.page_content) for c in semantic_chunks) / len(semantic_chunks):.0f} chars)")


### Chunking Strategy Comparison


In [ ]:
# Comparison statistics
comparison_df = pd.DataFrame({
    'Metric': ['Total Chunks', 'Avg Size', 'Min Size', 'Max Size'],
    'Standard (tokens)': [
        len(standard_chunks),
        int(sum(len(encoder.encode(c.page_content)) for c in standard_chunks) / len(standard_chunks)),
        min(len(encoder.encode(c.page_content)) for c in standard_chunks),
        max(len(encoder.encode(c.page_content)) for c in standard_chunks)
    ],
    'Semantic (chars)': [
        len(semantic_chunks),
        int(sum(len(c.page_content) for c in semantic_chunks) / len(semantic_chunks)),
        min(len(c.page_content) for c in semantic_chunks),
        max(len(c.page_content) for c in semantic_chunks)
    ]
})

print("\nChunking Comparison:")
print(comparison_df.to_string(index=False))


---

## Section 4: Vector Store Creation

Creating Qdrant vector stores for both chunking strategies. These will serve as the foundation for all retriever implementations.


### Vector Store 1: Standard Chunks


In [ ]:
# Create Qdrant vector store with standard chunks
vectorstore_standard = Qdrant.from_documents(
    standard_chunks,
    embeddings,
    location=":memory:",
    collection_name="standard_chunks"
)

print(f"Standard vector store created: {len(standard_chunks)} documents")


### Vector Store 2: Semantic Chunks


In [ ]:
# Create Qdrant vector store with semantic chunks
vectorstore_semantic = Qdrant.from_documents(
    semantic_chunks,
    embeddings,
    location=":memory:",
    collection_name="semantic_chunks"
)

print(f"Semantic vector store created: {len(semantic_chunks)} documents indexed")